In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf
import gspread
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

In [6]:
CONFIG_PATH = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
logistic_sheet_url = "https://docs.google.com/spreadsheets/d/1eo2tY_57lcOTtGOyU5GOz2IQON3b0VsVf8rTsbfT3HM/edit?gid=0#gid=0"
target_spread= "https://docs.google.com/spreadsheets/d/1hClSA4u_gE5KUudt2Vz2dHJVmh9a3AciCjycjB3Rvds/edit?gid=0#gid=0"
# target_spread = "https://docs.google.com/spreadsheets/d/190xi7j3lRGgOeK6-E88kMYVaDez16h2L35JdgM7n5mM/edit?gid=0#gid=0" (testing sheet url)
c = conf.get_config(file_name=CONFIG_PATH)
spread = Spread(logistic_sheet_url, config=c)
target_spread = Spread(target_spread,config=c)

In [7]:
try:
    df = spread.sheet_to_df(
        sheet="Order list", 
        index=None, 
    )
    print(f"Successfully loaded {len(df)} rows of agent data.")
except Exception as e:
    print(f"Error: {e}")

Successfully loaded 988 rows of agent data.


## SESSION WISE REPORT (AM & PM)

In [ ]:

def format_dates(df_input, cols):
    df_out = df_input.copy()
    for col in cols:
        df_out[col] = df_out[col].dt.strftime('%m/%d/%Y').where(df_out[col].notna(), other="")
    return df_out


def safe_upload(spread, sheet_name, df, create=True):
    spread.open_sheet(sheet_name, create=create)
    worksheet = spread.sheet

    worksheet.clear()

    required_rows = len(df) + 1  
    required_cols = len(df.columns)
    worksheet.resize(rows=max(required_rows, 2), cols=required_cols)

    headers = df.columns.tolist()
    worksheet.update(values=[headers], range_name='A1', value_input_option='USER_ENTERED')

    values = df.fillna("").values.tolist()
    if values:
        worksheet.update(values=values, range_name='A2', value_input_option='USER_ENTERED')

    print(f"Successfully updated '{sheet_name}' with {len(df)} rows.")


def get_report_session():

    now = datetime.now()
    hour = now.hour
    minute = now.minute

    if 10 <= hour < 12:
        return 'PENDING SHIPMENT (AM)'
    elif 12 <= hour <= 23:
        return 'PENDING SHIPMENT (PM)'
    else:
        return None


try:
    if df.empty:
        print("Data Frame is empty. No data to process.")
    else:
        print(f"Data Frame loaded with {len(df)} rows. Processing...")

        date_col = 'DATE'
        required_cols = {'STATUS', 'DELIVERY AGENT', date_col, 'DISPATCHED DATE', 'TRACKING NUMBER'}

        if not required_cols.issubset(df.columns):
            missing = required_cols - set(df.columns)
            print(f"Error: Missing columns in sheet: {missing}")
        else:

            # --- Clean Data ---
            df_clean = df.dropna(subset=['TRACKING NUMBER']).copy()
            df_clean = df_clean[df_clean['TRACKING NUMBER'].astype(str).str.strip() != ""]
            df_clean = df_clean[df_clean['STATUS'].astype(str).str.strip() != ""]

            df_clean[date_col] = pd.to_datetime(df_clean[date_col], dayfirst=True, errors='coerce')
            df_clean['DISPATCHED DATE'] = pd.to_datetime(df_clean['DISPATCHED DATE'], dayfirst=True, errors='coerce')

            date_cols = [date_col, 'DISPATCHED DATE']

            yesterday = pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
            today = pd.Timestamp.now().normalize()
            now = datetime.now()

            # --- TOTAL DISPATCH ---
            df_total_upload = format_dates(df_clean, date_cols)
            safe_upload(target_spread, 'TOTAL DISPATCH', df_total_upload)

            # --- TOTAL PENDING SHIPMENTS ---
            # Excludes: DELIVERED, DELIVERED AND UNPAID
            # mask_total_pending = (
            #     (~df_clean['STATUS'].astype(str).str.strip().str.upper().isin([
            #         "DELIVERED", "DELIVERED AND UNPAID"
            #     ])) &
            #     (df_clean[date_col] <= yesterday)
            # )
            # total_pending_df = df_clean[mask_total_pending].copy()
            # total_pending_upload = format_dates(total_pending_df, date_cols)

            # print(f"Filtered to {len(total_pending_upload)} rows. Uploading to 'TOTAL PENDING SHIPMENTS'...")
            # safe_upload(target_spread, 'TOTAL PENDING SHIPMENTS', total_pending_upload)

            # --- PENDING SHIPMENTS ---
            # Excludes: DELIVERED, DELIVERED AND UNPAID, RTO, CANCELLED
            # mask_pending = (
            #     (~df_clean['STATUS'].astype(str).str.strip().str.upper().isin([
            #         "DELIVERED", "DELIVERED AND UNPAID", "RTO", "CANCELLED"
            #     ])) &
            #     (df_clean[date_col] <= yesterday)
            # )
            mask_pending = (
                (~df_clean['STATUS'].astype(str).str.strip().str.upper().isin([
                    "DELIVERED", "DELIVERED AND UNPAID", "RTO", "CANCELLED"
                ])) 
            )
            pending_df = df_clean[mask_pending].copy()
            pending_upload = format_dates(pending_df, date_cols)

            print(f"Filtered to {len(pending_upload)} rows. Uploading to 'PENDING SHIPMENTS'...")
            # safe_upload(target_spread, 'PENDING SHIPMENTS', pending_upload)

            # --- AM / PM Pending Dispatch Report ---
            # session_sheet = get_report_session()

            # if session_sheet is None:
            #     print(f"Current time {now.strftime('%H:%M')} is outside reporting hours (10AM-11:59AM or 12PM onwards). Skipping AM/PM report.")
            # else:
            #     print(f"Current time {now.strftime('%H:%M')} → Uploading to '{session_sheet}'...")
            #     pending_dispatch_upload = format_dates(pending_df, date_cols)
            #     safe_upload(target_spread, session_sheet, pending_dispatch_upload)
            #     print(f"Successfully uploaded {len(pending_dispatch_upload)} rows to '{session_sheet}'.")

            # --- Age-Based Segmentation ---
            pending_df['AGE_DAYS'] = (today - pending_df['DISPATCHED DATE']).dt.days

            sheets_mapping = {
                '1-2 DAYS': pending_df[pending_df['AGE_DAYS'].between(1, 2)].copy(),
                '3-4 DAYS': pending_df[pending_df['AGE_DAYS'].between(3, 4)].copy(),
                '5-6 DAYS': pending_df[pending_df['AGE_DAYS'].between(5, 6)].copy(),
                '7 Days':   pending_df[pending_df['AGE_DAYS'] == 7].copy(),
                'Alert':    pending_df[pending_df['AGE_DAYS'] > 7].copy()
            }

            print(f"Summary for {today.date()}:")
            for sheet_name, filtered_df in sheets_mapping.items():
                try:
                    upload_segment = filtered_df.drop(columns=['AGE_DAYS'])
                    upload_segment = format_dates(upload_segment, date_cols)
                    safe_upload(target_spread, sheet_name, upload_segment)
                except Exception as e:
                    print(f"Error updating '{sheet_name}': {e}")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

Data Frame loaded with 988 rows. Processing...
Successfully updated 'TOTAL DISPATCH' with 864 rows.
Filtered to 525 rows. Uploading to 'PENDING SHIPMENTS'...
Summary for 2026-03-02:
Successfully updated '1-2 DAYS' with 126 rows.
Successfully updated '3-4 DAYS' with 101 rows.
Successfully updated '5-6 DAYS' with 46 rows.
Successfully updated '7 Days' with 32 rows.
Successfully updated 'Alert' with 206 rows.
